# שיעור 01 - מבוא לסוכני בינה מלאכותית

ברוכים הבאים לשיעור הראשון בקורס **סוכני בינה מלאכותית למתחילים**!

**סוכן בינה מלאכותית** הוא תוכנה שמשתמשת במודל שפה גדול (LLM) כמנוע ההסקה שלה ויכולה לבצע *פעולות* בעולם האמיתי — לקרוא ל-APIs, לשאול מסדי נתונים, או להריץ קוד — כדי להשיג מטרה עבור המשתמש.

בנוטבוק זה תבנו את הסוכן הראשון שלכם: **סוכן נסיעות** שממליץ על יעדי חופשה. לאורך הדרך תלמדו איך:

1. להתחבר לשירות סוכני Azure AI Foundry באמצעות **מסגרת הסוכנים של מיקרוסופט**.
2. לתת לסוכן **כלי** — פונקציית פייתון פשוטה שהוא יכול לקרוא לה.
3. להריץ את הסוכן ולבדוק את התגובה שלו.
4. להזרים את תגובת הסוכן טוקן-בטוקן.


## התקנה

לפני הפעלת המחברת הזו, וודא שיש לך:

1. **פרויקט Azure AI Foundry** עם מודל שיחה פרוס (למשל `gpt-4o-mini`).
2. **התחברת עם Azure CLI** — הפעל `az login` במסוף שלך.
3. **הגדרת את משתני הסביבה הנדרשים:**
   - `AZURE_AI_PROJECT_ENDPOINT` — נקודת הקצה של פרויקט Azure AI Foundry שלך.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — שם המודל שהפרסת.

התא התחתון מתקין את חבילת הפייתון שתצטרך.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from agent_framework import tool

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## יצירת הסוכן הראשון שלך

סוכן צריך שני דברים:

- **הוראות** שאומרות לו *מי הוא* ו*איך להתנהג* (מערכת הפעלה).
- **כלים** — פונקציות פייתון עם הקישוט `@tool` שהסוכן יכול לקרוא כדי לקבל מידע או לבצע פעולות.

להלן נגדיר כלי פשוט שמחזיר רשימה של יעדי חופשה פופולריים. הסוכן ישתמש בכלי זה כאשר משתמש יבקש המלצות לטיולים.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = provider.as_agent(
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
    tools=[get_destinations],
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## תשובות בזרם

לחוויה אינטראקטיבית יותר אתה יכול **לשדר** את תגובת הסוכן. במקום להמתין לתשובה מלאה, הסוכן מפיק קטעי טקסט בזמן שהם נוצרים. זה שימושי במיוחד בממשקי צ'אט שבהם רוצים להציג פלט בזמן אמת.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## סיכום

בשיעור זה למדת כיצד:

- **ליצור ספק** שמתחבר לשירות Azure AI Foundry Agent באמצעות `AzureAIProjectAgentProvider`.
- **להגדיר כלי** באמצעות הדקורטור `@tool` כך שהסוכן יוכל לקרוא לפונקציות הפייתון שלך.
- **להפעיל את הסוכן** עם הודעת משתמש ולהדפיס את התגובה שלו.
- **לשדר תגובות** לפלט בזמן אמת.

בשיעור הבא נחקר את מסגרות הסוכנים ביתר עומק ונלמד כיצד להעניק לסוכנים כלים עוצמתיים יותר ויכולות הסקת מסקנות מרובי שלבים.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**כתב ויתור**:
מסמך זה תורגם באמצעות שירות תרגום אוטומטי [Co-op Translator](https://github.com/Azure/co-op-translator). למרות שאנו שואפים לדיוק, יש לקחת בחשבון שתרגומים אוטומטיים עלולים להכיל שגיאות או אי-דיוקים. יש להחשיב את המסמך המקורי בשפתו הטבעית כמקור הסמכות. למידע קריטי מומלץ להשתמש בתרגום מקצועי על ידי מתרגם אדם. אנו לא אחראים לכל אי-הבנה או פירוש שגוי הנובע מהשימוש בתרגום זה.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
